# Этап 6: IrResnet4 A/B на dataset_v002 (+ референс v003)

**Цель:** два прогона на **dataset_v002** с протоколом `train_irresnet_original.yaml` (hidden=72, lr=1e-5, WRS, StepLR).

| Run | label_schema | Смысл |
|-----|--------------|-------|
| `colab06_v002_smarts` | `structure_smarts` | SMARTS совпал |
| `colab06_v002_structure` | `structure` | SMARTS и пик |

Референс: `colab06_irresnet_dataset_v003` на v003.

**Colab:** A + A2 + C. **Local:** B + C (`paths.local.yaml`; нужен v002 под processed_root).

Документация: [`docs/NOTEBOOKS.md`](../docs/NOTEBOOKS.md).

## Выбор среды (выполните ОДНУ ячейку)

| Среда | Запустить | Пропустить |
|-------|-----------|------------|
| **Google Colab** | **A. Colab** (+ при полном датасете **A2. Drive**) | **B. Local** |
| **Локальный Jupyter** | **B. Local** | **A** и **A2** (включая `drive.mount`) |

После A или B выполните **C. Пути и данные**.
Подробнее: [`docs/NOTEBOOKS.md`](../docs/NOTEBOOKS.md).


In [ ]:
# === A. Colab: окружение ===
# Локально эту ячейку НЕ запускайте (см. B. Local).
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Lamblador/IR_expert_system_3.git'
REPO_BRANCH = 'colab-v1'
REPO_DIR = Path('/content/IR_expert_system_3')

def _run_git(cmd, cwd=None):
    print('git', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if (REPO_DIR / '.git').is_dir():
    _run_git(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    _run_git(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    _run_git(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} существует, но это не git-репозиторий')
    _run_git([
        'git', 'clone', '-b', REPO_BRANCH, '--single-branch',
        REPO_URL, str(REPO_DIR),
    ])

ROOT = REPO_DIR.resolve()
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
rev = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, text=True
).strip()
print(f'ROOT: {ROOT} @ {REPO_BRANCH} ({rev})')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'], check=True)
print('pip install OK')
IR_ENV = 'colab'


### A2. Google Drive (только Colab full)

Пропустите для HF smoke и локально.

In [ ]:
# === A2. Colab Drive (full dataset) ===
# Нужен только для полного датасета на Google Drive.
# Для HF smoke (dataset_mini) эту ячейку ПРОПУСТИТЕ.
# Локально НЕ запускайте.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
IR_DATA = Path('/content/drive/MyDrive/ir_data')
RUNS_DRIVE = Path('/content/drive/MyDrive/ir_expert_system_3/runs')
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
print('IR_DATA exists:', IR_DATA.exists(), IR_DATA)
print('RUNS_DRIVE:', RUNS_DRIVE)


### B. Локальный Jupyter

Пропустите в Colab.

In [ ]:
# === B. Local: окружение ===
# В Google Colab эту ячейку НЕ запускайте (см. A. Colab).
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk-up до каталога с pyproject.toml (фикс nested-clone из notebooks/)."""
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').is_file():
            return p
    raise FileNotFoundError(
        'Не найден pyproject.toml выше cwd. '
        'Откройте ноутбук из клона репозитория или cd в корень IR_expert_system_3.'
    )

ROOT = _find_repo_root(Path.cwd())
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
print('ROOT (local, ветку не переключаем):', ROOT)

FORCE_REINSTALL = False  # True — принудительно pip install -e .[torch]
need_install = FORCE_REINSTALL
if not need_install:
    try:
        import ir_pipeline  # noqa: F401
    except ImportError:
        need_install = True
if need_install:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'],
        check=True,
    )
    print('pip install OK')
else:
    print('ir_pipeline уже установлен — pip пропущен (FORCE_REINSTALL=True для переустановки)')
IR_ENV = 'local'


### C. Пути и данные

После A или B.

In [ ]:
# === C. Пути и данные ===
# Выполните после A или B. Контракт: ROOT, PATHS_YAML, paths, DATASET_DIR, BANDS_YAML, RUNS_DIR
import os
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, resolve_paths

SMOKE_VERSION = 'dataset_mini'
FULL_VERSION = 'dataset_v003'
# Режим данных (если IR_ENV не задан — auto):
#   'local'       — configs/paths.local.yaml
#   'colab_smoke' — HF mini, paths.huggingface.yaml
#   'colab_full'  — Drive, paths.colab.yaml
DATA_MODE = 'auto'  # или 'local' | 'colab_smoke' | 'colab_full'

if 'IR_ENV' not in globals():
    IR_ENV = 'colab' if Path('/content').exists() else 'local'

if DATA_MODE == 'auto':
    if IR_ENV == 'local':
        DATA_MODE = 'local'
    elif 'IR_DATA' in globals() and Path(IR_DATA).exists():
        DATA_MODE = 'colab_full'
    else:
        DATA_MODE = 'colab_smoke'

if DATA_MODE == 'local':
    PATHS_YAML = Path('configs/paths.local.yaml')
    if not PATHS_YAML.is_file():
        raise FileNotFoundError(
            'Нет configs/paths.local.yaml — скопируйте configs/paths.local.example.yaml '
            'и пропишите raw_jcamp_dir / processed_root.'
        )
elif DATA_MODE == 'colab_full':
    if 'IR_DATA' not in globals():
        raise RuntimeError('Colab full: сначала выполните A2 (Drive mount) → IR_DATA')
    os.environ['IR_PROCESSED_ROOT'] = str(Path(IR_DATA) / 'processed')
    PATHS_YAML = Path('configs/paths.colab.yaml')
elif DATA_MODE == 'colab_smoke':
    PATHS_YAML = Path('configs/paths.huggingface.yaml')
else:
    raise ValueError(f'Неизвестный DATA_MODE={DATA_MODE!r}')

paths_cfg = load_yaml(PATHS_YAML)
if DATA_MODE == 'colab_smoke':
    paths_cfg['dataset_version'] = SMOKE_VERSION
    paths_cfg.pop('dataset_profile', None)
elif DATA_MODE == 'colab_full':
    paths_cfg['dataset_version'] = paths_cfg.get('dataset_version') or FULL_VERSION
# local: dataset_version / profile из yaml

paths = resolve_paths(paths_cfg)
DATASET_DIR = paths['processed_root'] / str(paths['dataset_version'])
BANDS_YAML = paths['bands_config']
RUNS_DIR = Path('runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

spectra = DATASET_DIR / 'spectra.npz'
if not spectra.is_file():
    raise FileNotFoundError(
        f'Нет {spectra}. DATA_MODE={DATA_MODE}, PATHS_YAML={PATHS_YAML}'
    )
print('DATA_MODE:', DATA_MODE)
print('PATHS_YAML:', PATHS_YAML)
print('DATASET_DIR:', DATASET_DIR)
print('dataset_version:', paths['dataset_version'])
print('raw_jcamp_dir:', paths['raw_jcamp_dir'])


### Фиксация dataset_v002 для A/B

In [ ]:
# Для A/B эксперимента фиксируем dataset_v002 (перекрывает C при необходимости)
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, resolve_paths

TARGET_VERSION = 'dataset_v002'
paths_cfg = load_yaml(PATHS_YAML)
paths_cfg['dataset_version'] = TARGET_VERSION
paths = resolve_paths(paths_cfg)
DATASET_DIR = paths['processed_root'] / TARGET_VERSION
BANDS_YAML = paths['bands_config']
assert (DATASET_DIR / 'spectra.npz').is_file(), f'Нет spectra.npz: {DATASET_DIR}'
assert (DATASET_DIR / 'labels_structure.parquet').is_file()
assert (DATASET_DIR / 'labels_structure_smarts.parquet').is_file()
if 'RUNS_DRIVE' not in globals() or RUNS_DRIVE is None:
    RUNS_DRIVE = Path('runs')  # local fallback
split_path = DATASET_DIR / 'split.json'
print('dataset:', DATASET_DIR)
print('PATHS_YAML:', PATHS_YAML)
print('split:', 'ok' if split_path.is_file() else 'нет — выполните dataset-split ниже')


## Split v2 на v002 (рекомендуется)

Без `val_ids` в split v1 код использует **test как val** — early stop и подбор порога нечестны относительно v003. Выполните один раз перед обучением (стратификация по `structure_smarts` достаточна для обоих прогонов).

In [ ]:
import subprocess
subprocess.run([
    'ir-pipeline', 'dataset-split',
    '--paths', str(PATHS_YAML),
    '--dataset-version', 'dataset_v002',
    '--label-schema', 'structure_smarts',
], check=True)
split_path = DATASET_DIR / 'split.json'
print(
    split_path.read_text(encoding='utf-8')[:400]
    if split_path.is_file() else 'split.json не создан'
)


In [ ]:
import pandas as pd
from ir_pipeline.dataset_preview import build_multilabel_matrix
from ir_pipeline.resnet_input import load_model_inputs

_, _, spec_ids, _, _ = load_model_inputs(DATASET_DIR)
bands = BANDS_YAML
rows = []
for schema in ['structure_smarts', 'structure', 'spectrum']:
    try:
        Y, _ = build_multilabel_matrix(DATASET_DIR, spec_ids, bands, label_schema=schema)
        rows.append({
            'schema': schema,
            'mean_labels': float(Y.sum(axis=1).mean()),
            'positives': int(Y.sum()),
        })
    except FileNotFoundError as e:
        rows.append({'schema': schema, 'error': str(e)})
pd.DataFrame(rows)

In [ ]:
%matplotlib inline
from copy import deepcopy
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults
from ir_pipeline.irresnet_train import IrResnetTrainer

base_cfg = merge_train_defaults(load_yaml(Path('configs/train_irresnet_original.yaml')))

RUNS = [
    ('structure_smarts', 'colab06_v002_smarts'),
    ('structure', 'colab06_v002_structure'),
]

summaries = []
run_dirs = {}
train_cfgs = {}
for schema, run_suffix in RUNS:
    cfg = deepcopy(base_cfg)
    cfg['label_schema'] = schema
    run_dir = Path('runs') / run_suffix
    print('===', schema, '=>', run_dir)
    trainer = IrResnetTrainer(
        dataset_dir=DATASET_DIR,
        run_dir=run_dir,
        bands_yaml=BANDS_YAML,
        train_cfg=cfg,
        label_schema=schema,
    )
    summary = trainer.fit()
    summary['run_suffix'] = run_suffix
    summaries.append(summary)
    run_dirs[run_suffix] = run_dir
    train_cfgs[run_suffix] = cfg
    print(summary)

In [ ]:
import json
import matplotlib.pyplot as plt
from ir_pipeline.train_monitor import IrResnetTrainingPlotter

for run_suffix, run_dir in run_dirs.items():
    cfg = train_cfgs[run_suffix]
    plotter = IrResnetTrainingPlotter.from_train_cfg(run_dir, cfg, title=f'IrResnet4 — {run_suffix}')
    hist_path = run_dir / 'irresnet_history.json'
    if hist_path.is_file():
        plotter.series = json.loads(hist_path.read_text(encoding='utf-8'))
        plotter.live_plot = True
        plotter._clear_and_plot(
            len(plotter.series.get('train_loss', [])),
            int(cfg.get('torch_epochs', 200)),
        )
    else:
        img = run_dir / 'irresnet_training_curve.png'
        if img.is_file():
            plt.figure(figsize=(10, 4))
            plt.imshow(plt.imread(img))
            plt.title(run_suffix)
            plt.axis('off')
            plt.show()

In [ ]:
import shutil
from pathlib import Path

for run_suffix, run_dir in run_dirs.items():
    run_name = f'{run_suffix}_dataset_v002'
    dest_root = RUNS_DRIVE if RUNS_DRIVE is not None else Path('runs')
    dest = Path(dest_root) / run_name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest)
    shutil.make_archive(str(Path(dest_root) / run_name), 'zip', run_dir)
    print('Saved:', dest)


## Сравнение с v003

Сводка по `irresnet_metrics.json`: v002 A/B + референс `colab06_irresnet_dataset_v003` (локально или на Drive).

In [ ]:
import json
import pandas as pd
from pathlib import Path

METRIC_KEYS = [
    'label_schema',
    'test_lrap',
    'test_f1_weighted',
    'test_f1_macro',
    'test_f1_micro',
    'val_lrap',
    'best_epoch',
]

def load_run_metrics(run_dir: Path) -> dict:
    p = run_dir / 'irresnet_metrics.json'
    if not p.is_file():
        return {'run': run_dir.name, 'error': 'metrics not found'}
    m = json.loads(p.read_text(encoding='utf-8'))
    row = {'run': run_dir.name}
    for k in METRIC_KEYS:
        row[k] = m.get(k)
    row['dataset'] = m.get('dataset_dir', str(run_dir))
    return row

# Локальные прогоны v002
compare_rows = [load_run_metrics(run_dirs[s]) for s in run_dirs]

# Референс v003: локально или на Drive
v003_candidates = [
    Path('runs/colab06_irresnet_dataset_v003'),
    RUNS_DRIVE / 'colab06_irresnet_dataset_v003',
    Path('../runs/colab06_irresnet_dataset_v003'),
]
for cand in v003_candidates:
    if (cand / 'irresnet_metrics.json').is_file():
        compare_rows.append(load_run_metrics(cand))
        break

pd.DataFrame(compare_rows)